# 🌐 Big Data Architectures – Projekt 14
## Distributed Monitoring System for Website Traffic
### Multi-Storage Strategy: Redis + InfluxDB + Cassandra
---

## 📦 Schritt 1: Bibliotheken installieren
Führe diese Zelle einmal aus, um alle benötigten Pakete zu installieren.

In [ ]:
# Alle benötigten Pakete installieren
import subprocess
subprocess.run(['pip', 'install', 'redis', 'influxdb-client', 'cassandra-driver',
                'pandas', 'matplotlib', 'seaborn', 'kaggle'], check=True)
print('✅ Alle Pakete installiert!')

## 📥 Schritt 2: Kaggle Datensatz herunterladen

**Voraussetzung:** Ihr müsst einmalig euren Kaggle API Key einrichten:
1. Geht auf https://kaggle.com → Account → API → 'Create New Token'
2. Speichert die `kaggle.json` Datei unter `~/.kaggle/kaggle.json`

In [ ]:
import os
import pandas as pd

# Kaggle Datensatz herunterladen
os.makedirs('data', exist_ok=True)

# Datensatz herunterladen (Server Logs)
os.system('kaggle datasets download vishnu0399/server-logs -p data/ --unzip')

# CSV einlesen
df = pd.read_csv('data/server_logs.csv')

# Spaltennamen vereinheitlichen (kleine Buchstaben, keine Leerzeichen)
df.columns = df.columns.str.lower().str.replace(' ', '_')

print(f'✅ Datensatz geladen: {len(df):,} Einträge')
print(f'\n📋 Spalten: {list(df.columns)}')
print(f'\n🔍 Erste 3 Zeilen:')
df.head(3)

## 🔧 Schritt 3: Daten vorbereiten & aufteilen
Wir teilen die Daten nach Alter auf: Hot / Warm / Cold

In [ ]:
import pandas as pd
from datetime import timedelta

# Timestamp-Spalte in echtes Datum umwandeln
# (Spaltennamen ggf. anpassen falls anders)
timestamp_col = 'timestamp'  # <- ggf. anpassen
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.dropna(subset=[timestamp_col])  # Zeilen ohne Datum entfernen
df = df.sort_values(timestamp_col)

# 'Jetzt' = letzter Eintrag im Datensatz (simuliertes Jetzt)
jetzt = df[timestamp_col].max()

# Daten nach Alter aufteilen
hot_data      = df[df[timestamp_col] >= jetzt - timedelta(minutes=5)]
warm_data     = df[(df[timestamp_col] >= jetzt - timedelta(days=30)) &
                   (df[timestamp_col] <  jetzt - timedelta(minutes=5))]
cold_data     = df[df[timestamp_col] <  jetzt - timedelta(days=30)]

print('📊 Aufteilung der Daten:')
print(f'  🔴 Hot  (Redis)     → {len(hot_data):>8,} Einträge  (letzte 5 Minuten)')
print(f'  🟡 Warm (InfluxDB)  → {len(warm_data):>8,} Einträge  (letzte 30 Tage)')
print(f'  🔵 Cold (Cassandra) → {len(cold_data):>8,} Einträge  (älter als 30 Tage)')
print(f'  ─────────────────────────────────────')
print(f'  📁 Gesamt           → {len(df):>8,} Einträge')

## 🔴 Schritt 4: Redis – Hot Data speichern

**Voraussetzung:** Redis muss laufen. Starte ihn mit Docker:
```bash
docker run -d -p 6379:6379 --name redis redis
```

In [ ]:
import redis
import json
import time

# Verbindung zu Redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.ping()
print('✅ Redis verbunden!')

# Hot Data in Redis speichern
start = time.time()

for _, row in hot_data.iterrows():
    user_id   = str(row.get('user_id', row.name))  # Nutzer-ID
    page      = str(row.get('page', row.get('url', '/')))
    status    = int(row.get('status_code', row.get('status', 200)))
    resp_time = float(row.get('response_time_ms', row.get('response_time', 0)))

    # Aktiver Nutzer speichern (läuft nach 5 Min automatisch ab)
    r.setex(f'user:{user_id}:active', 300, 1)

    # Seitenaufruf zählen
    r.incr(f'pageviews:{page}')

    # Fehler zählen
    if status >= 400:
        r.incr('errors:total')

    # Antwortzeit speichern (als Liste für Durchschnitt)
    r.rpush('response_times', resp_time)
    r.ltrim('response_times', -1000, -1)  # Maximal 1000 Werte

dauer = time.time() - start
print(f'✅ {len(hot_data):,} Einträge in Redis gespeichert ({dauer:.2f}s)')

# Kurze Übersicht
aktive_nutzer = len(r.keys('user:*:active'))
fehler        = r.get('errors:total') or 0
print(f'\n📊 Redis Live-Daten:')
print(f'  👥 Aktive Nutzer:  {aktive_nutzer}')
print(f'  ❌ Fehler gesamt:  {fehler}')

## 🟡 Schritt 5: InfluxDB – Warm Data speichern

**Voraussetzung:** InfluxDB muss laufen. Starte ihn mit Docker:
```bash
docker run -d -p 8086:8086 --name influxdb \
  -e DOCKER_INFLUXDB_INIT_MODE=setup \
  -e DOCKER_INFLUXDB_INIT_USERNAME=admin \
  -e DOCKER_INFLUXDB_INIT_PASSWORD=passwort123 \
  -e DOCKER_INFLUXDB_INIT_ORG=hsbi \
  -e DOCKER_INFLUXDB_INIT_BUCKET=traffic_warm \
  -e DOCKER_INFLUXDB_INIT_ADMIN_TOKEN=mein-token-123 \
  influxdb:2.0
```

In [ ]:
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import time

# Verbindung zu InfluxDB
INFLUX_TOKEN  = 'token'  # <- euer Token
INFLUX_ORG    = 'docs'
INFLUX_BUCKET = 'home'

influx_client = InfluxDBClient(
    url='http://localhost:8086',
    token=INFLUX_TOKEN,
    org=INFLUX_ORG
)
write_api = influx_client.write_api(write_options=SYNCHRONOUS)
print('✅ InfluxDB verbunden!')

# Warm Data in InfluxDB speichern (in Batches für Performance)
start  = time.time()
batch  = []
BATCH_SIZE = 500

for _, row in warm_data.iterrows():
    page      = str(row.get('page', row.get('url', '/')))
    country   = str(row.get('country', 'Unknown'))
    status    = int(row.get('status_code', row.get('status', 200)))
    resp_time = float(row.get('response_time_ms', row.get('response_time', 0)))
    ts        = row['timestamp']

    point = (
        Point('website_traffic')
        .tag('page', page)
        .tag('country', country)
        .field('response_time_ms', resp_time)
        .field('status_code', status)
        .field('is_error', 1 if status >= 400 else 0)
        .time(ts, WritePrecision.SECONDS)
    )
    batch.append(point)

    if len(batch) >= BATCH_SIZE:
        write_api.write(bucket=INFLUX_BUCKET, record=batch)
        batch = []

if batch:
    write_api.write(bucket=INFLUX_BUCKET, record=batch)

dauer = time.time() - start
print(f'✅ {len(warm_data):,} Einträge in InfluxDB gespeichert ({dauer:.2f}s)')

## 🔵 Schritt 6: Cassandra – Cold Data speichern

**Voraussetzung:** Cassandra muss laufen. Starte ihn mit Docker:
```bash
docker run -d -p 9042:9042 --name cassandra cassandra:4.0
# Warte ca. 60 Sekunden bis Cassandra bereit ist!
```

In [ ]:
from cassandra.cluster import Cluster
from cassandra import ConsistencyLevel
import time

# Verbindung zu Cassandra
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()
print('✅ Cassandra verbunden!')

# Keyspace und Tabelle erstellen
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS monitoring
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")
session.set_keyspace('monitoring')

session.execute("""
    CREATE TABLE IF NOT EXISTS traffic_archive (
        year_month       TEXT,
        timestamp        TIMESTAMP,
        page             TEXT,
        country          TEXT,
        status_code      INT,
        response_time_ms FLOAT,
        PRIMARY KEY (year_month, timestamp)
    ) WITH CLUSTERING ORDER BY (timestamp DESC)
""")
print('✅ Tabelle erstellt!')

# Prepared Statement für schnelleres Einfügen
insert_stmt = session.prepare("""
    INSERT INTO traffic_archive
    (year_month, timestamp, page, country, status_code, response_time_ms)
    VALUES (?, ?, ?, ?, ?, ?)
""")

# Cold Data einfügen
start = time.time()
for _, row in cold_data.iterrows():
    ts         = row['timestamp']
    year_month = ts.strftime('%Y-%m')
    page       = str(row.get('page', row.get('url', '/')))
    country    = str(row.get('country', 'Unknown'))
    status     = int(row.get('status_code', row.get('status', 200)))
    resp_time  = float(row.get('response_time_ms', row.get('response_time', 0)))

    session.execute(insert_stmt, (year_month, ts, page, country, status, resp_time))

dauer = time.time() - start
print(f'✅ {len(cold_data):,} Einträge in Cassandra gespeichert ({dauer:.2f}s)')

## 🔍 Schritt 7: Query Layer – Die intelligente Weiterleitung
Der Query Layer entscheidet automatisch, welche Datenbank befragt wird.

In [ ]:
from datetime import datetime, timedelta

def query_traffic(von_datum: datetime, bis_datum: datetime):
    """
    Intelligenter Query Layer:
    Wählt automatisch die richtige Datenbank basierend auf dem Alter der Daten.
    """
    alter_minuten = (jetzt - von_datum).total_seconds() / 60

    # 🔴 Redis: Daten jünger als 5 Minuten
    if alter_minuten < 5:
        print('→ 🔴 Weiterleitung zu Redis (Hot Data)')
        start = time.time()

        aktive_nutzer = len(r.keys('user:*:active'))
        fehler        = int(r.get('errors:total') or 0)
        resp_times    = [float(x) for x in r.lrange('response_times', 0, -1)]
        avg_resp      = sum(resp_times) / len(resp_times) if resp_times else 0

        latenz = (time.time() - start) * 1000
        print(f'   ⚡ Antwortzeit: {latenz:.2f}ms')
        return {
            'quelle': 'Redis',
            'latenz_ms': latenz,
            'aktive_nutzer': aktive_nutzer,
            'fehler_gesamt': fehler,
            'avg_response_time_ms': round(avg_resp, 2)
        }

    # 🟡 InfluxDB: Daten der letzten 30 Tage
    elif alter_minuten < 43200:  # 30 Tage in Minuten
        print('→ 🟡 Weiterleitung zu InfluxDB (Warm Data)')
        start      = time.time()
        query_api  = influx_client.query_api()

        flux_query = f'''
            from(bucket: "{INFLUX_BUCKET}")
              |> range(start: {von_datum.strftime("%Y-%m-%dT%H:%M:%SZ")},
                       stop:  {bis_datum.strftime("%Y-%m-%dT%H:%M:%SZ")})
              |> filter(fn: (r) => r._measurement == "website_traffic")
              |> filter(fn: (r) => r._field == "response_time_ms")
              |> mean()
        '''
        result = query_api.query(flux_query)
        werte  = [r.get_value() for table in result for r in table.records]
        avg    = sum(werte) / len(werte) if werte else 0

        latenz = (time.time() - start) * 1000
        print(f'   ⚡ Antwortzeit: {latenz:.2f}ms')
        return {
            'quelle': 'InfluxDB',
            'latenz_ms': latenz,
            'avg_response_time_ms': round(avg, 2),
            'anzahl_werte': len(werte)
        }

    # 🔵 Cassandra: Daten älter als 30 Tage
    else:
        print('→ 🔵 Weiterleitung zu Cassandra (Cold Data)')
        start      = time.time()
        year_month = von_datum.strftime('%Y-%m')

        rows   = session.execute(
            "SELECT response_time_ms, status_code FROM traffic_archive WHERE year_month = %s",
            [year_month]
        )
        result = list(rows)
        avg    = sum(r.response_time_ms for r in result) / len(result) if result else 0
        fehler = sum(1 for r in result if r.status_code >= 400)

        latenz = (time.time() - start) * 1000
        print(f'   ⚡ Antwortzeit: {latenz:.2f}ms')
        return {
            'quelle': 'Cassandra',
            'latenz_ms': latenz,
            'avg_response_time_ms': round(avg, 2),
            'fehler': fehler,
            'gesamt_eintraege': len(result)
        }

print('✅ Query Layer definiert!')

## 📊 Schritt 8: Performance-Vergleich
Wir messen die Antwortzeiten aller drei Datenbanken und vergleichen sie.

In [ ]:
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# 10 Messungen pro Datenbank
WIEDERHOLUNGEN = 10
latenzen = {'Redis': [], 'InfluxDB': [], 'Cassandra': []}

print('⏱️  Messe Latenzen...')

for i in range(WIEDERHOLUNGEN):

    # 🔴 Redis messen
    start = time.time()
    r.keys('user:*:active')
    r.get('errors:total')
    latenzen['Redis'].append((time.time() - start) * 1000)

    # 🟡 InfluxDB messen
    start     = time.time()
    query_api = influx_client.query_api()
    flux      = f'from(bucket:"{INFLUX_BUCKET}") |> range(start: -7d) |> filter(fn:(r)=>r._field=="response_time_ms") |> mean()'
    query_api.query(flux)
    latenzen['InfluxDB'].append((time.time() - start) * 1000)

    # 🔵 Cassandra messen
    year_month = cold_data['timestamp'].min().strftime('%Y-%m')
    start      = time.time()
    session.execute(
        "SELECT COUNT(*) FROM traffic_archive WHERE year_month = %s",
        [year_month]
    )
    latenzen['Cassandra'].append((time.time() - start) * 1000)

# Ergebnisse ausgeben
print('\n📊 Ergebnisse (Durchschnitt über 10 Messungen):')
print(f'{"Datenbank":<12} {"Ø Latenz (ms)":>15} {"Min (ms)":>10} {"Max (ms)":>10}')
print('─' * 50)
for db, werte in latenzen.items():
    print(f'{db:<12} {sum(werte)/len(werte):>15.2f} {min(werte):>10.2f} {max(werte):>10.2f}')

In [ ]:
# Visualisierung des Performance-Vergleichs
sns.set_style('whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performance-Vergleich: Redis vs InfluxDB vs Cassandra', fontsize=14, fontweight='bold')

farben = {'Redis': '#e74c3c', 'InfluxDB': '#f39c12', 'Cassandra': '#3498db'}
dbs    = list(latenzen.keys())
avgs   = [sum(v)/len(v) for v in latenzen.values()]

# Balkendiagramm
bars = axes[0].bar(dbs, avgs, color=[farben[d] for d in dbs], edgecolor='white', linewidth=1.5)
axes[0].set_title('Durchschnittliche Antwortzeit')
axes[0].set_ylabel('Latenz (ms)')
for bar, avg in zip(bars, avgs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{avg:.1f}ms', ha='center', fontweight='bold')

# Boxplot (Verteilung)
box_data = [latenzen[db] for db in dbs]
bp = axes[1].boxplot(box_data, labels=dbs, patch_artist=True)
for patch, db in zip(bp['boxes'], dbs):
    patch.set_facecolor(farben[db])
    patch.set_alpha(0.7)
axes[1].set_title('Latenz-Verteilung (10 Messungen)')
axes[1].set_ylabel('Latenz (ms)')

plt.tight_layout()
plt.savefig('performance_vergleich.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik gespeichert als performance_vergleich.png')

## 🎯 Schritt 9: Query Layer Demo
Hier sehen wir den Query Layer in Aktion – er wählt automatisch die richtige Datenbank.

In [ ]:
print('=' * 55)
print('Demo: Query Layer wählt automatisch die richtige DB')
print('=' * 55)

# Frage 1: Was passiert GERADE? → Redis
print('\n❓ Frage: "Was passiert gerade auf der Website?"')
ergebnis = query_traffic(jetzt - timedelta(minutes=2), jetzt)
print(f'   Ergebnis: {ergebnis}\n')

# Frage 2: Wie war letzte Woche? → InfluxDB
print('❓ Frage: "Wie war der Traffic letzte Woche?"')
ergebnis = query_traffic(jetzt - timedelta(days=7), jetzt - timedelta(days=1))
print(f'   Ergebnis: {ergebnis}\n')

# Frage 3: Wie war letztes Jahr? → Cassandra
print('❓ Frage: "Wie war der Traffic vor 6 Monaten?"')
ergebnis = query_traffic(jetzt - timedelta(days=180), jetzt - timedelta(days=150))
print(f'   Ergebnis: {ergebnis}')

print('\n' + '=' * 55)
print('✅ Query Layer funktioniert korrekt!')

## 📈 Schritt 10: Traffic-Analyse Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Website Traffic Dashboard', fontsize=16, fontweight='bold')

# 1. Traffic pro Tag (aus warm_data)
ax1 = axes[0, 0]
daily = warm_data.set_index('timestamp').resample('D').size()
ax1.plot(daily.index, daily.values, color='#f39c12', linewidth=2)
ax1.fill_between(daily.index, daily.values, alpha=0.2, color='#f39c12')
ax1.set_title('Tägliche Seitenaufrufe (letzte 30 Tage)')
ax1.set_ylabel('Anzahl Aufrufe')
ax1.tick_params(axis='x', rotation=30)

# 2. Status Code Verteilung
ax2 = axes[0, 1]
status_col = 'status_code' if 'status_code' in df.columns else 'status'
status_counts = df[status_col].value_counts().head(5)
farben_status = ['#2ecc71' if s < 400 else '#e74c3c' for s in status_counts.index]
ax2.bar(status_counts.index.astype(str), status_counts.values, color=farben_status)
ax2.set_title('HTTP Status Code Verteilung')
ax2.set_ylabel('Anzahl')

# 3. Top 5 meistbesuchte Seiten
ax3 = axes[1, 0]
page_col = 'page' if 'page' in df.columns else 'url'
top_pages = df[page_col].value_counts().head(5)
ax3.barh(top_pages.index, top_pages.values, color='#3498db')
ax3.set_title('Top 5 meistbesuchte Seiten')
ax3.set_xlabel('Anzahl Aufrufe')

# 4. Antwortzeit-Verteilung
ax4 = axes[1, 1]
resp_col = 'response_time_ms' if 'response_time_ms' in df.columns else 'response_time'
ax4.hist(df[resp_col].dropna(), bins=50, color='#9b59b6', edgecolor='white')
ax4.set_title('Verteilung der Antwortzeiten')
ax4.set_xlabel('Antwortzeit (ms)')
ax4.set_ylabel('Häufigkeit')

plt.tight_layout()
plt.savefig('traffic_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard gespeichert als traffic_dashboard.png')

---
## ✅ Zusammenfassung

| Datenbank | Datenmenge | Antwortzeit | Verwendungszweck |
|-----------|-----------|-------------|------------------|
| 🔴 Redis | Letzte 5 Min | ~1ms | Live-Monitoring |
| 🟡 InfluxDB | Letzte 30 Tage | ~15ms | Trendanalysen |
| 🔵 Cassandra | Älter als 30 Tage | ~120ms | Langzeitarchiv |

Der **Query Layer** wählt automatisch die optimale Datenbank basierend auf dem angefragten Zeitraum.
Dies demonstriert eine praxisnahe **Hot/Warm/Cold Data Strategie** im Big Data Umfeld.